# Layered Yelp Review Prediction — Pretrained Embeddings + Soft Routing

This script implements a **two-layer modeling approach** for predicting Yelp star ratings:
1. **Sentiment Model:** Predicts coarse-grained sentiment (negative / neutral / positive)
2. **Star Rating Model:** Uses both pretrained embeddings and *soft sentiment probabilities* to predict the detailed 1–5 star rating. By using **pretrained sentence embeddings** (`all-MiniLM-L6-v2`) from `sentence-transformers` and **soft routing** (passing sentiment probabilities rather than
hard labels), this model captures nuanced sentiment signals while avoiding error propagation.

### Pipeline Overview
- Load cleaned Yelp data
- Encode text using pretrained sentence embeddings
- Train a logistic regression sentiment classifier
- Append sentiment probabilities to embeddings
- Train a second logistic regression for 5-star rating prediction

### Output
- Sentiment classifier performance
- Star rating classifier performance

In [6]:
import os
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score

In [7]:
folder_name = 'extracted-features'

# Load embeddings and metadata
X = np.load(os.path.join(folder_name, "sentencetransformer_embeddings.npy"))
meta = pd.read_csv(os.path.join(folder_name, "sentencetransformer_meta.csv"))

sample_size = min(500000, len(meta))
data_idx = meta.sample(sample_size, random_state=42).index

X_sample = X[data_idx]
y_sent = meta.loc[data_idx, 'sentiment'].values
y_star = meta.loc[data_idx, 'stars'].values

# Train/test split
X_train, X_test, y_sent_train, y_sent_test = train_test_split(X_sample, y_sent, test_size=0.1, random_state=42, stratify=y_sent)
_, _, y_star_train, y_star_test = train_test_split(X_sample, y_star, test_size=0.1, random_state=42, stratify=y_star)


In [8]:
# Layer 1 — Sentiment Classifier
print("\nTraining sentiment classifier...")
sentiment_clf = LogisticRegression(max_iter=1000)
sentiment_clf.fit(X_train, y_sent_train)

y_pred_sent = sentiment_clf.predict(X_test)
print("\n=== Sentiment Classification Report ===")
print(classification_report(y_sent_test, y_pred_sent, digits=3))
print(f"Sentiment Accuracy: {accuracy_score(y_sent_test, y_pred_sent):.4f}")

# Get soft sentiment probabilities
train_sent_probs = sentiment_clf.predict_proba(X_train) 
test_sent_probs = sentiment_clf.predict_proba(X_test) 


Training sentiment classifier...

=== Sentiment Classification Report ===
              precision    recall  f1-score   support

           0      0.764     0.791     0.778     10246
           1      0.450     0.158     0.234      5654
           2      0.867     0.951     0.907     34100

    accuracy                          0.829     50000
   macro avg      0.694     0.633     0.640     50000
weighted avg      0.799     0.829     0.804     50000

Sentiment Accuracy: 0.8287


In [9]:
# Layer 2 — Star Rating Classifier
print("\nTraining star rating classifier with soft sentiment routing...")
alpha = 1.0  # weight for sentiment features — can tune

X_train_star = np.hstack([X_train, alpha * train_sent_probs])
X_test_star = np.hstack([X_test, alpha * test_sent_probs])

star_clf = LogisticRegression(max_iter=1000)
star_clf.fit(X_train_star, y_star_train)

y_pred_star = star_clf.predict(X_test_star)
print("\n=== Star Rating Classification Report ===")
print(classification_report(y_star_test, y_pred_star, digits=3))
print(f"Star Rating Accuracy: {accuracy_score(y_star_test, y_pred_star):.4f}")


Training star rating classifier with soft sentiment routing...

=== Star Rating Classification Report ===
              precision    recall  f1-score   support

         1.0      0.000     0.000     0.000      6078
         2.0      0.000     0.000     0.000      4169
         3.0      0.000     0.000     0.000      5653
         4.0      0.000     0.000     0.000     11866
         5.0      0.445     1.000     0.616     22234

    accuracy                          0.445     50000
   macro avg      0.089     0.200     0.123     50000
weighted avg      0.198     0.445     0.274     50000

Star Rating Accuracy: 0.4447


/Users/juliasober/anaconda3/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1469: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/Users/juliasober/anaconda3/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1469: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/Users/juliasober/anaconda3/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1469: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
